In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

try:
    from statsmodels.stats.proportion import proportions_ztest, proportion_confint
except:
    ! pip install statsmodels
    from statsmodels.stats.proportion import proportions_ztest, proportion_confint

## Helper Classes

In [ ]:
class FrequentistABTest:
    def __init__(self, str_uri, str_dirname_output, flt_alpha=0.05):
        self.str_uri = str_uri
        self.str_dirname_output = str_dirname_output
        self.flt_alpha = flt_alpha
        self.df_data = None
        self.dict_results = {}

    def import_data(self):
        self.df_data = pd.read_parquet(self.str_uri)
        print(f'Data loaded: {self.df_data.shape[0]:,} rows')
        return self

    def run_z_test(self, str_metric):
        df_control = self.df_data[self.df_data['version'] == 'gate_30']
        df_treatment = self.df_data[self.df_data['version'] == 'gate_40']
        int_n_control = len(df_control)
        int_n_treatment = len(df_treatment)
        int_successes_control = df_control[str_metric].sum()
        int_successes_treatment = df_treatment[str_metric].sum()
        flt_rate_control = int_successes_control / int_n_control
        flt_rate_treatment = int_successes_treatment / int_n_treatment
        flt_z_stat, flt_p_value = proportions_ztest(
            count=[int_successes_control, int_successes_treatment],
            nobs=[int_n_control, int_n_treatment],
            alternative='two-sided'
        )
        flt_diff = flt_rate_treatment - flt_rate_control
        flt_se = np.sqrt(
            flt_rate_control * (1 - flt_rate_control) / int_n_control +
            flt_rate_treatment * (1 - flt_rate_treatment) / int_n_treatment
        )
        flt_z_crit = stats.norm.ppf(1 - self.flt_alpha / 2)
        flt_ci_lower = flt_diff - flt_z_crit * flt_se
        flt_ci_upper = flt_diff + flt_z_crit * flt_se
        str_day = '1-Day' if str_metric == 'retention_1' else '7-Day'
        print(f'\n=== Z-TEST: {str_day} Retention ===')
        print(f'Control (gate_30) rate: {flt_rate_control:.4f} ({int_successes_control:,}/{int_n_control:,})')
        print(f'Treatment (gate_40) rate: {flt_rate_treatment:.4f} ({int_successes_treatment:,}/{int_n_treatment:,})')
        print(f'Difference: {flt_diff:.4f} ({flt_diff*100:.2f} percentage points)')
        print(f'Z-statistic: {flt_z_stat:.4f}')
        print(f'P-value: {flt_p_value:.6f}')
        print(f'95% CI for difference: [{flt_ci_lower:.4f}, {flt_ci_upper:.4f}]')
        bool_significant = flt_p_value < self.flt_alpha
        print(f'Significant at alpha={self.flt_alpha}: {bool_significant}')
        self.dict_results[str_metric] = {
            'str_metric': str_day,
            'flt_rate_control': flt_rate_control,
            'flt_rate_treatment': flt_rate_treatment,
            'flt_diff': flt_diff,
            'flt_z_stat': flt_z_stat,
            'flt_p_value': flt_p_value,
            'flt_ci_lower': flt_ci_lower,
            'flt_ci_upper': flt_ci_upper,
            'bool_significant': bool_significant
        }
        return self

    def run_chi_squared_test(self, str_metric):
        df_contingency = pd.crosstab(self.df_data['version'], self.df_data[str_metric])
        flt_chi2, flt_p_value, int_dof, arr_expected = stats.chi2_contingency(df_contingency)
        str_day = '1-Day' if str_metric == 'retention_1' else '7-Day'
        print(f'\n=== CHI-SQUARED TEST: {str_day} Retention ===')
        print(f'Contingency Table:')
        print(df_contingency)
        print(f'\nChi-squared statistic: {flt_chi2:.4f}')
        print(f'P-value: {flt_p_value:.6f}')
        print(f'Degrees of freedom: {int_dof}')
        print(f'Significant at alpha={self.flt_alpha}: {flt_p_value < self.flt_alpha}')
        return self

    def run_mann_whitney_test(self):
        ser_control = self.df_data.loc[self.df_data['version'] == 'gate_30', 'sum_gamerounds']
        ser_treatment = self.df_data.loc[self.df_data['version'] == 'gate_40', 'sum_gamerounds']
        flt_u_stat, flt_p_value = stats.mannwhitneyu(ser_control, ser_treatment, alternative='two-sided')
        print(f'\n=== MANN-WHITNEY U TEST: Game Rounds ===')
        print(f'Control (gate_30) median: {ser_control.median():.0f}')
        print(f'Treatment (gate_40) median: {ser_treatment.median():.0f}')
        print(f'Control (gate_30) mean: {ser_control.mean():.2f}')
        print(f'Treatment (gate_40) mean: {ser_treatment.mean():.2f}')
        print(f'U-statistic: {flt_u_stat:,.0f}')
        print(f'P-value: {flt_p_value:.6f}')
        print(f'Significant at alpha={self.flt_alpha}: {flt_p_value < self.flt_alpha}')
        self.dict_results['gamerounds'] = {
            'str_metric': 'Game Rounds',
            'flt_median_control': ser_control.median(),
            'flt_median_treatment': ser_treatment.median(),
            'flt_u_stat': flt_u_stat,
            'flt_p_value': flt_p_value,
            'bool_significant': flt_p_value < self.flt_alpha
        }
        return self

    def plot_confidence_intervals(self):
        list_metrics = []
        list_diffs = []
        list_ci_lower = []
        list_ci_upper = []
        for str_key in ['retention_1', 'retention_7']:
            if str_key in self.dict_results:
                dict_r = self.dict_results[str_key]
                list_metrics.append(dict_r['str_metric'])
                list_diffs.append(dict_r['flt_diff'] * 100)
                list_ci_lower.append(dict_r['flt_ci_lower'] * 100)
                list_ci_upper.append(dict_r['flt_ci_upper'] * 100)
        fig, ax = plt.subplots(figsize=(10, 5))
        arr_y = np.arange(len(list_metrics))
        list_errors = [[d - l for d, l in zip(list_diffs, list_ci_lower)],
                        [u - d for d, u in zip(list_diffs, list_ci_upper)]]
        ax.errorbar(list_diffs, arr_y, xerr=list_errors, fmt='o', color='#4C72B0',
                    markersize=10, capsize=8, capthick=2, linewidth=2)
        ax.axvline(x=0, color='red', linestyle='--', label='No Effect')
        ax.set_yticks(arr_y)
        ax.set_yticklabels(list_metrics)
        ax.set_xlabel('Difference in Retention (Percentage Points)')
        ax.set_title('95% Confidence Intervals for Treatment Effect', fontsize=14, y=1.02)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3, axis='x')
        for i, (flt_d, flt_l, flt_u) in enumerate(zip(list_diffs, list_ci_lower, list_ci_upper)):
            ax.annotate(f'{flt_d:.2f}pp [{flt_l:.2f}, {flt_u:.2f}]',
                        xy=(flt_d, i), xytext=(0, 15), textcoords='offset points',
                        ha='center', fontsize=10)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_confidence_intervals.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_retention_comparison(self):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for idx, str_key in enumerate(['retention_1', 'retention_7']):
            if str_key in self.dict_results:
                dict_r = self.dict_results[str_key]
                list_rates = [dict_r['flt_rate_control'] * 100, dict_r['flt_rate_treatment'] * 100]
                list_labels = ['gate_30 (Control)', 'gate_40 (Treatment)']
                list_colors = ['#4C72B0', '#DD8452']
                bars = axes[idx].bar(list_labels, list_rates, color=list_colors, edgecolor='black')
                axes[idx].set_title(f'{dict_r["str_metric"]} Retention Rates', fontsize=14, y=1.02)
                axes[idx].set_ylabel('Retention Rate (%)')
                flt_p = dict_r['flt_p_value']
                str_sig = f'p = {flt_p:.4f}' if flt_p >= 0.0001 else f'p < 0.0001'
                if dict_r['bool_significant']:
                    str_sig += ' *'
                axes[idx].text(0.5, 0.95, str_sig, transform=axes[idx].transAxes,
                               ha='center', fontsize=12, style='italic')
                for bar, flt_val in zip(bars, list_rates):
                    axes[idx].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
                                   f'{flt_val:.2f}%', ha='center', fontsize=11)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_retention_comparison.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_p_value_summary(self):
        list_tests = []
        list_p_values = []
        for str_key, dict_r in self.dict_results.items():
            list_tests.append(dict_r['str_metric'])
            list_p_values.append(dict_r['flt_p_value'])
        fig, ax = plt.subplots(figsize=(10, 5))
        list_colors = ['#C44E52' if p < self.flt_alpha else '#4C72B0' for p in list_p_values]
        ax.barh(list_tests, list_p_values, color=list_colors, edgecolor='black')
        ax.axvline(x=self.flt_alpha, color='red', linestyle='--', label=f'alpha = {self.flt_alpha}')
        ax.set_xlabel('P-value')
        ax.set_title('P-values Across All Tests', fontsize=14, y=1.02)
        ax.legend(fontsize=11)
        for i, flt_p in enumerate(list_p_values):
            str_label = f'{flt_p:.4f}' if flt_p >= 0.0001 else f'{flt_p:.2e}'
            ax.text(flt_p + 0.002, i, str_label, va='center', fontsize=11)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/03_p_value_summary.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def save_results(self):
        list_rows = []
        for str_key, dict_r in self.dict_results.items():
            dict_row = {
                'metric': dict_r['str_metric'],
                'p_value': dict_r['flt_p_value'],
                'significant': dict_r['bool_significant']
            }
            if 'flt_diff' in dict_r:
                dict_row['difference'] = dict_r['flt_diff']
                dict_row['ci_lower'] = dict_r['flt_ci_lower']
                dict_row['ci_upper'] = dict_r['flt_ci_upper']
            list_rows.append(dict_row)
        df_results = pd.DataFrame(list_rows)
        df_results.to_csv(f'{self.str_dirname_output}/frequentist_results.csv', index=False)
        print(f'\nResults saved to {self.str_dirname_output}/frequentist_results.csv')
        print(df_results.to_string(index=False))
        return self

## Constants

In [ ]:
str_bucket = 'ab-testing-demo-repo'
str_task = 'frequentist_test'
str_dirname_output = './output'
str_uri = f's3://{str_bucket}/00_data_collection/cookie_cats.parquet'
flt_alpha = 0.05

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f'Output directory ready: {str_dirname_output}')

## Run Frequentist Tests

In [ ]:
cls_test = FrequentistABTest(str_uri, str_dirname_output, flt_alpha=flt_alpha)
cls_test.import_data()

In [ ]:
cls_test.run_z_test('retention_1')

In [ ]:
cls_test.run_chi_squared_test('retention_1')

In [ ]:
cls_test.run_z_test('retention_7')

In [ ]:
cls_test.run_chi_squared_test('retention_7')

In [ ]:
cls_test.run_mann_whitney_test()

In [ ]:
cls_test.plot_confidence_intervals()

In [ ]:
cls_test.plot_retention_comparison()

In [ ]:
cls_test.plot_p_value_summary()

In [ ]:
cls_test.save_results()

## Completion

In [ ]:
print('\n=== FREQUENTIST TESTING COMPLETE ===')
print(f'Results and visualizations saved to: {str_dirname_output}')